In [1]:
import pymupdf
import os
import sys
from cloudinary import uploader, config
import json
from typing import Dict
import re

# make sure the project root is on the import path
sys.path.append('/home/inacio/script-clima-amazonia')

from helpers.texts.dictClasses import dict_classes
from helpers.texts.slugify import slugify
from helpers.images.analysis import img_analysis
from helpers.images.current_conditions import img_current_conditions
from helpers.images.multimodel import img_multimodel
from helpers.images.reference import img_reference
from helpers.images.categorization import img_categorization
from helpers.images.anomaly import img_anomaly

# Text

In [2]:
def date_to_en(date):
    meses = {
        "janeiro": "January",
        "fevereiro": "February",
        "março": "March",
        "marco": "March",
        "abril": "April",
        "maio": "May",
        "junho": "June",
        "julho": "July",
        "agosto": "August",
        "setembro": "September",
        "outubro": "October",
        "novembro": "November",
        "dezembro": "December"
    }
    mes = date.split()[2]
    dia = date.split()[0]
    ano = date.split()[-1]
    mes_en = meses[mes]
    date_en = f'{mes_en} {dia}, {ano}'
    return date_en

def date_to_es(date):
    meses = {
        "janeiro": "enero",
        "fevereiro": "febrero",
        "março": "marzo",
        "marco": "marzo",
        "abril": "abril",
        "maio": "mayo",
        "junho": "junio",
        "julho": "julio",
        "agosto": "agosto",
        "setembro": "septiembre",
        "outubro": "octubre",
        "novembro": "noviembre",
        "dezembro": "diciembre"
    }
    mes = date.split()[2]
    dia = date.split()[0]
    ano = date.split()[-1]
    mes_es = meses[mes]
    date_es = f'{dia} de {mes_es} de {ano}'
    return date_es

def get_meta(doc, bulletin_dict):
    # Page 1
    page = doc.load_page(0)
    text = page.get_text()
    text_lines = text.splitlines()
    number = text_lines[-1].split()[3]
    date = " ".join(text_lines[-1].split()[5:])

    meta = {
        "meta": {
        "pt": {
            "doi": f'10.61818/029106{number}',
            "issn": "2965-0291",
            "volume": "06",
            "date": date
        },
        "en": {
            "doi": f'10.61818/785704{number}',
            "issn": "2965-7857",
            "volume": "04",
            "date": date_to_en(date)
        },
        "es": {
            "doi": f'10.61818/770904{number}',
            "issn": "2965-7709",
            "volume": "04",
            "date": date_to_es(date)
        },
         "number": number
    }
        }
    bulletin_dict.update(meta)
    return bulletin_dict

def get_current_conditions(pathPT, pathEN, pathES, bulletin_dict):
    doc = pymupdf.open(pathPT)     
    page = doc.load_page(3)
    textPT = page.get_text()
    textPT = re.sub(r'\n+', ' ', textPT)
    textPT = re.sub(r'\s+', ' ', textPT)
    textPT = textPT.split("Condições atuais ")[1]
    textPT = textPT.split("1 Abacaxis ")[0].rstrip()
    # EN
    doc = pymupdf.open(pathEN)     
    page = doc.load_page(3)
    textEN = page.get_text()
    textEN = re.sub(r'\n+', ' ', textEN)
    textEN = re.sub(r'\s+', ' ', textEN)
    textEN = textEN.split("Current conditions ")[1]
    textEN = textEN.split("1 Abacaxis ")[0].rstrip()
    # ES
    doc = pymupdf.open(pathES)     
    page = doc.load_page(3)
    textES = page.get_text()
    textES = re.sub(r'\n+', ' ', textES)
    textES = re.sub(r'\s+', ' ', textES)
    textES = textES.split("Condiciones actuales ")[1]
    textES = textES.split("1 Abacaxis ")[0].rstrip()
    
    current_conditions = {
        "pt": textPT,
        "en": textEN,
        "es": textES
    }
    
    bulletin_dict['current_conditions'] = current_conditions
    return bulletin_dict 


def extract_fields(text: str) -> Dict[str, str]:

    climatologia = re.search(r"entre\s+(\d+\s+e\s+\d+\s+mm)", text).group(1)
    min = climatologia.split()[0]
    max = climatologia.split()[2]
    observados = re.search(r"foram\s+observados\s+(\d+\s+mm)", text)
    anomalia = re.search(r"valor\s+de\s+([-\d\.]+)", text)
    classification = re.search(r"classifica\s+a\s+bacia\s+em\s+condição\s+de\s+([^\.]+)", text)
    prognostico = re.search(r"sugere\s+um\s+comportamento\s+([^\.]+)", text)
    trend = text.split("comportamento climático indica ")
    trend = trend[1].split(" ",1)[0]


    return {
        "min": min,
        "max": max,
        "observados": observados.group(1) if observados else None,
        "anomalia": anomalia.group(1) if anomalia else None,
        "classification": classification.group(1).strip() if classification else None,
        "prognostico": prognostico.group(1).strip() if prognostico else None,
        "trend": trend
    }


def dict_trend(trend, lang):
    d = {
        "elevacao": {
            "en": "increase",
            "es": "elevación"
        },
        "manutencao": {
            "en": "increase",
            "es": "maintenance"
        },
        "reducao": {
            "en": "decrease",
            "es": "reducción"
        }
    }
    return d[trend][lang]

def dict_prognostico(prognostico, lang):
    d = {
        "muito-seco-ou-tendencia-a-extremamente-seco": {
            "en": "very dry behavior or a tending to be extremely dry",
            "es": "muy seco o con tendencia a ser extremadamente seco"
        },
        "muito-seco-ou-tendencia-a-muito-seco": {
            "en": "very dry behavior or a tending to be very dry",
            "es": "muy seco o con tendencia a ser muy seco"
        },
        "seco-ou-tendencia-a-muito-seco": {
            "en": "dry behavior or a tending to be very dry",
            "es": "seco o con tendencia a ser muy seco"
        },
        "chuvoso-ou-tendencia-a-chuvoso": {
            "en": "rainy behavior or a tending to be rainy",
            "es": "lluvioso o propenso a lluvioso."
        },
        "proximo-da-normalidade-ou-tendencia-a-chuvoso": {
            "en": "behavior close to normality or a tending to be rainy",
            "es": "cerca de la normalidad o con tendencia a lluvioso"
        },
        "proximo-da-normalidade-ou-tendencia-a-seco": {
            "en": "behavior close to normality or a tending to be dry",
            "es": "cerca de la normalidad o con tendencia a ser seco"
        },
         "proximo-da-normalidade": {
            "en": "behavior close to normality",
            "es": "cerca de la normalidad"
        },
         "seco-ou-tendencia-a-seco": {
            "en": "dry behavior or a tending to be dry",
            "es": "seco o con tendencia a ser seco"
        },
         'chuvoso-ou-tendencia-a-muito-chuvoso': {
            "en": "rainy behavior or a tending to be very rainy",
            "es": "lluvioso o con tendencia a ser muy lluvioso"
        },
         'muito-chuvoso-ou-tendencia-a-muito-chuvoso': {
            "en": "very rainy behavior or a tending to be very rainy",
            "es": "muy lluvioso o con tendencia a muy lluvioso"
        },
         'muito-chuvoso-ou-tendencia-a-extremamente-chuvoso': {
            "en": "very rainy behavior or a tending to be extremely rainy",
            "es": "muy seco o con tendencia a ser extremadamente seco"
        },
         'extremamente-chuvoso-ou-tendencia-a-extremamente-chuvoso': {
            "en": "extremely rainy behavior or a tending to be extremely rainy",
            "es": "extremadamente lluvioso o con tendencia a extremadamente lluvioso"
        }
         
         
         
    }
    return d[prognostico][lang]  

In [ ]:
def get_meta_2025(doc, bulletin_dict):
    # Page 1
    page = doc.load_page(0)
    text = page.get_text()
    text_lines = text.splitlines()
    for i in text_lines:
        if "Número" in i:
            number = i.split()[-1]
            break
        
    for i in text_lines:
        if "Manaus" in i:
            date = i.replace("Manaus, ", "").lstrip()
            break

    meta = {
        "meta": {
        "pt": {
            "doi": f'10.61818/029105{number}',
            "issn": "2965-0291",
            "volume": "05",
            "date": date
        },
        "en": {
            "doi": f'10.61818/785705{number}',
            "issn": "2965-7857",
            "volume": "03",
            "date": date_to_en(date)
        },
        "es": {
            "doi": f'10.61818/770903{number}',
            "issn": "2965-7709",
            "volume": "03",
            "date": date_to_es(date)
        },
         "number": number
    }
        }
    bulletin_dict.update(meta)
    return bulletin_dict

In [4]:
def get_analysis(doc, bulletin_dict):
    bacias = ['Bacia do Rio Branco', 'Bacia do Rio Negro','Bacia do Rio Marañon', 'Bacia do Rio Ucayali', 'Bacia do Rio Napo', 
'Curso principal do Rio Amazonas (Peru)','Bacia do Rio Javari','Bacia do Rio Içá (Putumayo)','Bacia do Rio Jutaí','Bacia do Rio Juruá',
'Bacia do Rio Japurá (Caquetá)','Bacia do Rio Tefé','Bacia do Rio Coari','Bacia do Rio Purus','Curso principal do Rio Solimões',
'Bacia dos rios Beni e Madre de Dios','Bacia do Rio Mamoré','Bacia do Rio Guaporé (Iténez)','Bacia do Rio Ji-Paraná','Bacia do Rio Aripuanã',
'Bacia do Rio Madeira','Bacias da margem esquerda do Rio Amazonas (Amazonas)','Bacia do Rio Abacaxis','Bacia do Rio Juruena','Bacia do Rio Teles Pires',
'Bacia do Rio Tapajós','Bacias da margem esquerda do Rio Amazonas (noroeste do Pará)','Bacia do Rio Curuá Una','Bacias da margem esquerda do Rio Amazonas (nordeste do PA)',
'Bacia do Rio Iriri','Bacia do Rio Xingu','Curso principal do Rio Amazonas (Brasil)']
    analysis = []
    texts = [doc.load_page(pg).get_text() for pg in range(4,15)]
    text = " ".join(texts)
    text = re.sub(r"\s*\n\s*", " ", text)
    text = re.sub(r"\s{2,}", " ", text).strip()
    for i, nome in enumerate(bacias):
        nome_regex = re.escape(nome)
        start_match = re.search(nome_regex, text)
        start = start_match.end()
        if i < len(bacias) - 1:
            next_nome = re.escape(bacias[i + 1])
            next_match = re.search(next_nome, text)
            end = next_match.start() if next_match else len(text)
        else:
            end = len(text)
        basin_text = text[start:end].strip()
        fields = extract_fields(basin_text)
        slug_name = slugify(nome)
        classification = fields["classification"]
        slug_classification = slugify(classification)
        # trend
        trend = fields["trend"]
        slug_trend = slugify(trend)
        # prognostico
        prognostico = fields["prognostico"]
        slug_prognostico = slugify(prognostico)

        
        bacia = {
                "id": slug_name,
                "name": nome,
                "min": fields["min"],
                "max": fields["max"],
                "observados": fields["observados"],
                "anomalia": fields["anomalia"],
                "i18n": {
                    "pt": {
                        "classification": classification,
                        "trend": trend,
                        "prognostico": prognostico
                    },
                    "en": {
                        "classification": dict_classes(slug_classification, "en"),
                        "trend": dict_trend(slug_trend, "en"),
                        "prognostico": dict_prognostico(slug_prognostico, "en")
                    },
                    "es": {
                        "classification": dict_classes(slug_classification, "es"),
                        "trend": dict_trend(slug_trend, "es"),
                        "prognostico": dict_prognostico(slug_prognostico, "es")
                    }
                }
            }
        # print(bacia)
        analysis.append(bacia)
    bulletin_dict['analysis'] = analysis
    
    return bulletin_dict

In [5]:
def get_multimodel(pathPT, pathEN, pathES, bulletin_dict):
    multimodel = {
        "pt": {},
        "en": {},
        "es": {}
    }
    # pt
    doc = pymupdf.open(pathPT)
    page = doc.load_page(15)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        if "A Figura acima," in text:
            seven_days = text.replace("\n", "")
            multimodel['pt']['seven_days'] = seven_days
            break
    page = doc.load_page(16)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        if "A Figura acima," in text:
            fourteen_days = text.replace("\n", "")
            multimodel['pt']['fourteen_days'] = fourteen_days
            break
    # en
    doc = pymupdf.open(pathEN)
    page = doc.load_page(15)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        text = re.sub(r'\s+', ' ', text).strip()
        if "The figure above"in text:
            seven_days = text.replace("\n", "")
            multimodel['en']['seven_days'] = seven_days
            break
    page = doc.load_page(16)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        text = re.sub(r'\s+', ' ', text).strip()
        if "The figure above" in text:
            fourteen_days = text.replace("\n", "")
            multimodel['en']['fourteen_days'] = fourteen_days
            break
    # es
    doc = pymupdf.open(pathES)
    page = doc.load_page(15)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        text = re.sub(r'\s+', ' ', text).strip()
        if "La figura de arriba"in text:
            seven_days = text.replace("\n", "")
            multimodel['es']['seven_days'] = seven_days
            break
    page = doc.load_page(16)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        text = re.sub(r'\s+', ' ', text).strip()
        if "La figura de arriba" in text:
            fourteen_days = text.replace("\n", "")
            multimodel['es']['fourteen_days'] = fourteen_days
            break

    bulletin_dict['multimodel'] = multimodel


    return bulletin_dict

In [6]:
def get_text(pathPT, yyyy, mmdd):
    bulletin_dict = {}
    doc = pymupdf.open(pathPT)
    # bulletin_dict = get_meta(doc, bulletin_dict)
    bulletin_dict = get_meta_2025(doc, bulletin_dict)    
    bulletin_dict = get_current_conditions(pathPT, pathEN, pathES, bulletin_dict)
    bulletin_dict = get_analysis(doc, bulletin_dict)
    bulletin_dict = get_multimodel(pathPT, pathEN, pathES, bulletin_dict)
    with open(f'/home/inacio/script-clima-amazonia/data/{yyyy}/{mmdd}.json', 'w') as f:
        json.dump(bulletin_dict, f, ensure_ascii=False, indent=3)
    return bulletin_dict   



In [15]:
mes = 'jan'
mmdd = '0115'
yyyy = "2025"
number = f"{yyyy}{mmdd}"
pathPT = f"/home/inacio/script-clima-amazonia/pdfs/BHA_PT_{number}.pdf"
pathEN = f"/home/inacio/script-clima-amazonia/pdfs/BHA_EN_{number}.pdf"
pathES = f"/home/inacio/script-clima-amazonia/pdfs/BHA_ES_{number}.pdf"

bulletin_dict = get_text(pathPT, yyyy, mmdd, mes)

# Images

In [7]:
def get_images(pathPT, output_path):
    doc = pymupdf.open(pathPT)
    img_current_conditions(doc, output_path)
    img_analysis(doc, output_path)
    img_multimodel(doc, output_path)
    img_reference(doc, output_path)
    img_categorization(doc, output_path)
    img_anomaly(doc, output_path)

In [8]:
output_path = f"/home/inacio/script-clima-amazonia/data/images/{yyyy}/{mes}/{mmdd}"
os.mkdir(output_path)
os.mkdir(f'{output_path}/analysis')
os.mkdir(f'{output_path}/anomaly')
os.mkdir(f'{output_path}/categorization')
os.mkdir(f'{output_path}/current_conditions')
os.mkdir(f'{output_path}/multimodel')
os.mkdir(f'{output_path}/reference')

NameError: name 'yyyy' is not defined

In [18]:
get_images(pathPT, output_path)

In [16]:
config(
  cloud_name="dveg6vhbi",
  api_key="954532419258163",
  api_secret="MiQDehQG2afKCxVi-QC8qMFE0Ag"
)

In [19]:
# path = f'{output_path}{edition}'
d_imgs = {}
for section in os.listdir(output_path):
    full_path = f'{output_path}/{section}'
    d_imgs[section] = {}
    imgs = os.listdir(full_path)
    for img in imgs:
        full_img_path = f'{full_path}/{img}'
        img_name = img.split(".")[0]
        result = uploader.upload(
            full_img_path,
            public_id=img_name,
            folder=f"clima-amazonia/{yyyy}/{mes}/{mmdd}/{section}",
            use_filename=True,
            overwrite=True,
            unique_filename=True
        )
        url = result['secure_url']
        d_imgs[section][img_name] = url
        print(result)

{'asset_id': '30d44b8c0ebc4b894f205bb7bba605ea', 'public_id': 'clima-amazonia/2025/jan/0115/current_conditions/table_current_conditions', 'version': 1773339439, 'version_id': 'ab283dc47a552a94c7ac0249fbb89ac5', 'signature': 'f0452cc30b6099cd647acefcc5295e9bf4db86bc', 'width': 1269, 'height': 253, 'format': 'png', 'resource_type': 'image', 'created_at': '2026-03-12T18:17:19Z', 'tags': [], 'bytes': 65995, 'type': 'upload', 'etag': '1f083b003fc6fba8ad7ed1e70b4731e4', 'placeholder': False, 'url': 'http://res.cloudinary.com/dveg6vhbi/image/upload/v1773339439/clima-amazonia/2025/jan/0115/current_conditions/table_current_conditions.png', 'secure_url': 'https://res.cloudinary.com/dveg6vhbi/image/upload/v1773339439/clima-amazonia/2025/jan/0115/current_conditions/table_current_conditions.png', 'asset_folder': 'clima-amazonia/2025/jan/0115/current_conditions', 'display_name': 'table_current_conditions', 'original_filename': 'table_current_conditions', 'api_key': '954532419258163'}
{'asset_id': 'a

In [9]:
eds = """0219
0226
0305
0312
0319
0326
0402
0409
0416
0423
0430
0507
0514
0521
0528
0604
0611
0618
0625
0702
0709
0716
0723
0730
0806
0813
0820
0827
0903
0910
0917
0924
1001
1008
1015
1022
1029
1105
1112
1119
1126
1203
1210
1217
1224
1231"""

In [10]:
eds = eds.splitlines()

In [47]:
mes = 'fev'
mmdd = '0205'
yyyy = "2025"
number = f"{yyyy}{mmdd}"
pathPT = f"/home/inacio/script-clima-amazonia/pdfs/BHA_PT_{number}.pdf"
pathEN = f"/home/inacio/script-clima-amazonia/pdfs/BHA_EN_{number}.pdf"
pathES = f"/home/inacio/script-clima-amazonia/pdfs/BHA_ES_{number}.pdf"

bulletin_dict = get_text(pathPT, yyyy, mmdd, mes)

In [27]:
yyyy = "2025"
for ed in eds:
    number = f"{yyyy}{ed}"
    pathPT = f"/home/inacio/script-clima-amazonia/pdfs/BHA_PT_{number}.pdf"
    pathEN = f"/home/inacio/script-clima-amazonia/pdfs/BHA_EN_{number}.pdf"
    pathES = f"/home/inacio/script-clima-amazonia/pdfs/BHA_ES_{number}.pdf"
    bulletin_dict = get_text(pathPT, yyyy, ed)
    print(bulletin_dict)

{'meta': {'pt': {'doi': '10.61818/02910608', 'issn': '2965-0291', 'volume': '05', 'date': '19 de fevereiro de 2025'}, 'en': {'doi': '10.61818/78570408', 'issn': '2965-7857', 'volume': '03', 'date': 'February 19, 2025'}, 'es': {'doi': '10.61818/77090408', 'issn': '2965-7709', 'volume': '03', 'date': '19 de febrero de 2025'}, 'number': '08'}, 'current_conditions': {'pt': 'Mapas das condições observadas de precipitação e gráficos individuais por bacias são produzidos a partir dos dados MERGE/GPM gerados pelo INPE/CPTEC, considerando como climatologia o período de 2000 a 2024. Entre os dias 21 de janeiro a 19 de fevereiro de 2025, permanece o quadro de chuvas abaixo da climatologia na porção oeste da área monitorada com deficit de precipitação sobre as bacias hidrográficas dos rios Guaporé, Japurá, Mamoré, Napo, Purus e Teles Pires, chuvas acima da climatologia caracterizaram as bacias do rios Abacaxis, Aripuanã, Branco, Iriri, Ji-Paraná, Jutaí, Madeira, Marañon, bacias da margem esquerda 

KeyError: 'Número'

In [15]:
for i in os.listdir("./data/2025"):
    mmdd = i.split(".")[0]
    output_path = f"/home/inacio/script-clima-amazonia/data/images/{yyyy}/{mmdd}"
    os.mkdir(output_path)
    os.mkdir(f'{output_path}/analysis')
    os.mkdir(f'{output_path}/anomaly')
    os.mkdir(f'{output_path}/categorization')
    os.mkdir(f'{output_path}/current_conditions')
    os.mkdir(f'{output_path}/multimodel')
    os.mkdir(f'{output_path}/reference')
    number = f"{yyyy}{mmdd}"
    pathPT = f"/home/inacio/script-clima-amazonia/pdfs/BHA_PT_{number}.pdf"
    pathEN = f"/home/inacio/script-clima-amazonia/pdfs/BHA_EN_{number}.pdf"
    pathES = f"/home/inacio/script-clima-amazonia/pdfs/BHA_ES_{number}.pdf"
    get_images(pathPT, output_path)
    print(mmdd)

MuPDF error: format error: No common ancestor in structure tree

0416
0723
0604
0319
0423
0219
0702
0507
0409
0226
0514
0716
0709
0326
0205
0521
0312
0305
0528
0212
0806
0618
MuPDF error: format error: No common ancestor in structure tree

0402
0611
0730
0625
0430


In [ ]:
output_path = f"/home/inacio/script-clima-amazonia/data/images/{yyyy}/{mmdd}"
os.mkdir(output_path)
os.mkdir(f'{output_path}/analysis')
os.mkdir(f'{output_path}/anomaly')
os.mkdir(f'{output_path}/categorization')
os.mkdir(f'{output_path}/current_conditions')
os.mkdir(f'{output_path}/multimodel')
os.mkdir(f'{output_path}/reference')

get_images(pathPT, output_path)

In [40]:
mmdd = "0226"
number = f"{yyyy}{mmdd}"
pathPT = f"/home/inacio/script-clima-amazonia/pdfs/BHA_PT_{number}.pdf"
pathEN = f"/home/inacio/script-clima-amazonia/pdfs/BHA_EN_{number}.pdf"
pathES = f"/home/inacio/script-clima-amazonia/pdfs/BHA_ES_{number}.pdf"
d_imgs = {}
output_path = f"/home/inacio/script-clima-amazonia/data/images/{yyyy}/{mmdd}"
for section in os.listdir(output_path):
    full_path = f'{output_path}/{section}'
    d_imgs[section] = {}
    imgs = os.listdir(full_path)
    for img in imgs:
        full_img_path = f'{full_path}/{img}'
        img_name = img.split(".")[0]
        result = uploader.upload(
            full_img_path,
            public_id=img_name,
            folder=f"clima-amazonia/{yyyy}/{mmdd}/{section}",
            use_filename=True,
            overwrite=True,
            unique_filename=True
        )
        url = result['secure_url']
        d_imgs[section][img_name] = url
        print(result)
        
pdfs = {}
result = uploader.upload(
            pathPT,
            public_id=pathPT.split("/")[-1].replace(".pdf",""),
            folder=f"clima-amazonia/{yyyy}/{mmdd}/",
            use_filename=True,
            overwrite=True,
            unique_filename=True
        )
pdfs['pt'] = result['secure_url']

result = uploader.upload(
            pathEN,
            public_id=pathEN.split("/")[-1].replace(".pdf",""),
            folder=f"clima-amazonia/{yyyy}/{mmdd}/",
            use_filename=True,
            overwrite=True,
            unique_filename=True
        )
pdfs['en'] = result['secure_url']

result = uploader.upload(
            pathES,
            public_id=pathES.split("/")[-1].replace(".pdf",""),
            folder=f"clima-amazonia/{yyyy}/{mmdd}/",
            use_filename=True,
            overwrite=True,
            unique_filename=True
        )
pdfs['es'] = result['secure_url']

{'asset_id': '6019e6c942251541a0ba2f83d53a1197', 'public_id': 'clima-amazonia/2025/0226/current_conditions/table_current_conditions', 'version': 1773348479, 'version_id': '3be53c01e78cc1f678e4e5714c747d8d', 'signature': '1567ffbb4f182729088abec7efdd1e3966198270', 'width': 1269, 'height': 253, 'format': 'png', 'resource_type': 'image', 'created_at': '2026-03-12T20:47:59Z', 'tags': [], 'bytes': 65705, 'type': 'upload', 'etag': '3798de636739d783de894fcecd05006d', 'placeholder': False, 'url': 'http://res.cloudinary.com/dveg6vhbi/image/upload/v1773348479/clima-amazonia/2025/0226/current_conditions/table_current_conditions.png', 'secure_url': 'https://res.cloudinary.com/dveg6vhbi/image/upload/v1773348479/clima-amazonia/2025/0226/current_conditions/table_current_conditions.png', 'asset_folder': 'clima-amazonia/2025/0226/current_conditions', 'display_name': 'table_current_conditions', 'original_filename': 'table_current_conditions', 'api_key': '954532419258163'}
{'asset_id': '591b7f6e130835ca7

In [41]:
jsonPath = f'./data/2025/{mmdd}.json'
with open(jsonPath, 'r') as f:
    bulletin_dict = json.load(f)
    
bulletin_dict['images'] = d_imgs
bulletin_dict['pdf'] = pdfs
with open(jsonPath, 'w') as f:
        json.dump(bulletin_dict, f, ensure_ascii=False, indent=3)

# Issues

In [21]:
from PIL import Image

In [30]:
path = "/home/inacio/script-clima-amazonia/data/issues/2025"
for i in os.listdir(path):
    imgPath = f'{path}/{i}'
    img = Image.open(imgPath)
    img_resize = img.resize((450, 500))
    img_resize.save(imgPath)
    print(i)

0122.png
0108.png
0129.png
0115.png
0212.png
0101.png
0205.png


In [ ]:
for folder in imgs:
    makedirs(f'resize/{folder}', exist_ok=True)
    c = 1
    for img in listdir(f'./capas/{folder}'):
        img = Image.open(f'./capas/{folder}/{img}')
        img_redimensionada = img.resize((200, 300))
        img_redimensionada.save(f'./resize/{folder}/{c}.jpg')
        c += 1

# Verificações

In [2]:
path = '/home/inacio/clima-amazonia/data/boletim/2025'

In [12]:
for i in os.listdir(path):
    jsonPath = f'{path}/{i}'
    with open(jsonPath, 'r') as fr:
        boletim = json.load(fr)
        fr.close()
        
    vol = boletim['meta']['pt']['volume']
    print(vol)
    if vol == '06':
        boletim['meta']['pt']['volume'] = '05'
        with open(jsonPath, 'w') as fw:
            json.dump(boletim, fw, ensure_ascii=False, indent=3)
                
            

06
06
05
05
06
06
06
06
06


In [10]:
boletim

{'meta': {'pt': {'doi': '10.61818/02910604',
   'issn': '2965-0291',
   'volume': '05',
   'date': '22 de janeiro de 2025'},
  'en': {'doi': '10.61818/78570404',
   'issn': '2965-7857',
   'volume': '04',
   'date': 'January 22, 2025'},
  'es': {'doi': '10.61818/77090404',
   'issn': '2965-7709',
   'volume': '04',
   'date': '22 de enero de 2025'},
  'number': '04'},
 'current_conditions': {'pt': 'Mapas das condições observadas de precipitação e gráficos individuais por bacias são produzidos a partir dos dados MERGE/GPM gerados pelo INPE/CPTEC, considerando como climatologia o período de 2000 a 2024. Entre os dias 24 de dezembro a 22 de janeiro de 2025, permanece o quadro de chuvas abaixo da climatologia em grande parte da área monitorada com deficit de precipitação sobre o curso principal do Rio Amazonas em território peruano, as bacias hidrográficas dos rios Branco, Içá, Japurá, Javari, Juruá, Jutaí, Marañon, bacias da margem esquerda do Rio Amazonas no nordeste do Estado do Amazona